# 04 — Embeddings visuales y textuales con CLIP

Este notebook reutiliza el preprocesamiento del notebook 03 y extrae las representaciones visuales `static` y `gripper`, junto con las textuales de CLIP. El encoder se importa desde `src/` y, al estar congelado, sus embeddings pueden cachearse para el entrenamiento posterior del modelo VLA.

## 1. Configuración e imports

Se definen las variables editables y se instancia `CLIPEncoder`. La arquitectura y la carga de CLIP permanecen en `src/clip_encoder.py`; aquí solo se crea el objeto y se usan sus interfaces públicas.

In [1]:
from pathlib import Path
import json
import os
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

# Permitimos importar los módulos del proyecto cuando se ejecuta desde notebooks/.
ROOT_DIR = Path(os.getcwd()).resolve().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.clip_encoder import CLIPEncoder
from src.dataset import VLADataset
from src.project_config import PROJECT_DIR, CACHE_DIR as SHARED_CACHE_DIR, taco_play_dir, ensure_project_dirs

# Variables editables del experimento.
RUTA = taco_play_dir()
NOMBRE_MODELO_CLIP = 'ViT-B-32'
PESOS_PREENTRENADOS = 'openai'
CONGELAR_CLIP = True
DISPOSITIVO = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_HILOS_CPU = os.cpu_count() or 1
BATCH_SIZE = 4
# Mayor lote solo para inferencia congelada; reducirlo si falta memoria RAM.
BATCH_SIZE_CACHE = 16
NORMALIZAR_EMBEDDINGS = True
CACHEAR_EMBEDDINGS = True
RUTA_CACHE = SHARED_CACHE_DIR

if DISPOSITIVO.type == 'cpu':
    torch.set_num_threads(NUM_HILOS_CPU)

# El encoder expone preprocess y tokenizer para construir el Dataset.
encoder = CLIPEncoder(NOMBRE_MODELO_CLIP, PESOS_PREENTRENADOS, DISPOSITIVO, CONGELAR_CLIP)
print(f'Modelo CLIP: {NOMBRE_MODELO_CLIP} ({PESOS_PREENTRENADOS})')
print(f'Dispositivo: {DISPOSITIVO}')
print(f'Hilos CPU para inferencia: {torch.get_num_threads()}')
# Rutas compartidas: no dependen del directorio desde el que se abra el notebook.
ROOT_DIR = PROJECT_DIR
ensure_project_dirs()
RUTA = taco_play_dir()


c:\TFM_Codigo\TFM-VLA\.venv\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Modelo CLIP: ViT-B-32 (openai)
Dispositivo: cpu
Hilos CPU para inferencia: 8


## 2. Carga exclusiva de las particiones definitivas

Se cargan únicamente las tablas generadas por el notebook 03. No se reconstruye el dataset ni se vuelve a dividir: si faltan los archivos o los identificadores requeridos, hay que ejecutar primero el notebook 03.

Las rutas y los fotogramas locales de `static` y `gripper` ya fueron calculados en el notebook 03 a partir de los límites físicos de los vídeos. Este notebook solo reutiliza esas particiones.

In [2]:
RUTA_PROCESADO = ROOT_DIR / 'data' / 'procesado_clip'
RUTA_PARAMETROS = ROOT_DIR / 'data' / 'parametros_normalizacion.json'
nombres = {'train': 'train.parquet', 'validation': 'validation.parquet', 'test': 'test.parquet'}
rutas_particiones = {clave: RUTA_PROCESADO / nombre for clave, nombre in nombres.items()}
if not all(ruta.exists() for ruta in rutas_particiones.values()):
    raise FileNotFoundError('Faltan train.parquet, validation.parquet o test.parquet. Ejecuta primero 03_preprocesamiento.ipynb.')
if not RUTA_PARAMETROS.exists():
    raise FileNotFoundError(f'No existe {RUTA_PARAMETROS}. Ejecuta primero 03_preprocesamiento.ipynb.')

particiones = {clave: pd.read_parquet(ruta) for clave, ruta in rutas_particiones.items()}
identificadores = {'episode_index', 'frame_index', 'task_index', 'index'}
columnas_vistas = {'imagen_static', 'fotograma_static', 'imagen_gripper', 'fotograma_gripper'}
for clave, tabla in particiones.items():
    faltantes = (identificadores | columnas_vistas) - set(tabla.columns)
    if faltantes:
        raise ValueError(f'{clave} no procede de 03: faltan columnas {sorted(faltantes)}')
    if tabla['index'].duplicated().any():
        raise ValueError(f'{clave} contiene índices de muestra repetidos')
print(f'Particiones cargadas exclusivamente desde {RUTA_PROCESADO}')

with open(RUTA_PARAMETROS, encoding='utf-8') as archivo:
    parametros = json.load(archivo)
minimo_accion = np.asarray(parametros['minimo'], dtype=np.float32)
escala_accion = np.asarray(parametros['escala'], dtype=np.float32)
def normalizar_accion(accion):
    return (np.asarray(accion, dtype=np.float32) - minimo_accion) / escala_accion

tablas = {clave: tabla.copy() for clave, tabla in particiones.items()}
for tabla in tablas.values():
    tabla['imagen_static'] = tabla['imagen_static'].astype(str)
    tabla['imagen_gripper'] = tabla['imagen_gripper'].astype(str)
    tabla['accion'] = tabla['accion'].apply(normalizar_accion)
datasets = {clave: VLADataset(tabla, encoder.preprocess, encoder.tokenizer) for clave, tabla in tablas.items()}
loaders = {'train': DataLoader(datasets['train'], batch_size=BATCH_SIZE, shuffle=True, num_workers=0), 'validation': DataLoader(datasets['validation'], batch_size=BATCH_SIZE, shuffle=False, num_workers=0), 'test': DataLoader(datasets['test'], batch_size=BATCH_SIZE, shuffle=False, num_workers=0)}
print({clave: len(ds) for clave, ds in datasets.items()})


Particiones cargadas exclusivamente desde C:\TFM_Codigo\TFM-VLA\data\procesado_clip
{'train': 190212, 'validation': 23760, 'test': 23826}


## 3. Embeddings visuales

Se toma un lote y se pasan por separado las vistas `static` y `gripper` al encoder visual. Cada resultado contiene un vector por imagen; con `ViT-B/32` la dimensión esperada es 512.

In [3]:
imagenes_static_batch, imagenes_gripper_batch, tokens_batch, acciones_batch = next(iter(loaders['train']))
embeddings_static = encoder.encode_image(imagenes_static_batch)
embeddings_gripper = encoder.encode_image(imagenes_gripper_batch)
print(f'Entrada static: {tuple(imagenes_static_batch.shape)}')
print(f'Embedding static: {tuple(embeddings_static.shape)}')
print(f'Entrada gripper: {tuple(imagenes_gripper_batch.shape)}')
print(f'Embedding gripper: {tuple(embeddings_gripper.shape)}')
assert embeddings_static.shape[0] == imagenes_static_batch.shape[0]
assert embeddings_gripper.shape[0] == imagenes_gripper_batch.shape[0]
if NOMBRE_MODELO_CLIP == 'ViT-B-32':
    assert embeddings_static.shape[1] == embeddings_gripper.shape[1] == 512
    print('La dimensión 512 coincide con ViT-B/32.')

Entrada static: (4, 3, 224, 224)
Embedding static: (4, 512)
Entrada gripper: (4, 3, 224, 224)
Embedding gripper: (4, 512)
La dimensión 512 coincide con ViT-B/32.


## 4. Embedding textual

Se reutilizan los tokens del mismo lote. La dimensión coincide con la visual porque ambas modalidades se proyectan al mismo espacio de CLIP.

In [4]:
embeddings_texto = encoder.encode_text(tokens_batch)
print(f'Entrada textual: {tuple(tokens_batch.shape)}')
print(f'Embedding textual: {tuple(embeddings_texto.shape)}')
assert embeddings_texto.shape == embeddings_static.shape == embeddings_gripper.shape
print('Las dimensiones visuales y textual coinciden.')

Entrada textual: (4, 77)
Embedding textual: (4, 512)
Las dimensiones visuales y textual coinciden.


## 5. Comprobación conjunta: similitud coseno

Se compara por separado cada vista con su instrucción y, opcionalmente, con un texto desplazado dentro del lote. La media de los pares correctos debería ser superior si el lote contiene instrucciones informativas.

In [5]:
static_norm = F.normalize(embeddings_static.float(), dim=-1)
gripper_norm = F.normalize(embeddings_gripper.float(), dim=-1)
textos_norm = F.normalize(embeddings_texto.float(), dim=-1)
similitudes_static = (static_norm * textos_norm).sum(dim=-1)
similitudes_gripper = (gripper_norm * textos_norm).sum(dim=-1)
if len(textos_norm) > 1:
    # El desplazamiento crea pares negativos sencillos sin cambiar el lote.
    textos_aleatorios = textos_norm.roll(shifts=1, dims=0)
    aleatorias_static = (static_norm * textos_aleatorios).sum(dim=-1)
    aleatorias_gripper = (gripper_norm * textos_aleatorios).sum(dim=-1)
    print(f'Static — pares correctos: {similitudes_static.mean().item():.4f} | aleatorios: {aleatorias_static.mean().item():.4f}')
    print(f'Gripper — pares correctos: {similitudes_gripper.mean().item():.4f} | aleatorios: {aleatorias_gripper.mean().item():.4f}')
else:
    print(f'Similitud static: {similitudes_static.item():.4f}')
    print(f'Similitud gripper: {similitudes_gripper.item():.4f}')

Static — pares correctos: 0.2112 | aleatorios: 0.2109
Gripper — pares correctos: 0.2519 | aleatorios: 0.2432


## 6. Cacheo de embeddings

Como CLIP está congelado, sus salidas no cambian entre épocas. Se recorren todos los lotes y se guarda un `.npz` por partición con embeddings `static`, `gripper`, textuales y acciones para ahorrar cómputo en el transformer.

Para evitar el coste del acceso aleatorio, las tres particiones se ordenan por `index` y se decodifican en un unico recorrido secuencial de los MP4. Los embeddings se separan de nuevo por particion antes de guardarse.

In [6]:
def extraer_embeddings_secuenciales(tablas):
    """Extrae las tres particiones en un solo recorrido secuencial de los MP4."""
    tabla_cache = pd.concat(
        [tabla.assign(particion=nombre) for nombre, tabla in tablas.items()],
        ignore_index=True,
    ).sort_values("index").reset_index(drop=True)

    dataset_cache = VLADataset(
        tabla_cache, encoder.preprocess, encoder.tokenizer, lectura_secuencial=True
    )
    loader_cache = DataLoader(
        dataset_cache, batch_size=BATCH_SIZE_CACHE, shuffle=False, num_workers=0
    )
    acumulados = {nombre: {"imagenes_static": [], "imagenes_gripper": [], "textos": [], "acciones": []}
                  for nombre in tablas}
    inicio = 0
    instrucciones_unicas = tabla_cache["instruccion"].drop_duplicates().tolist()
    tokens_unicos = encoder.tokenizer(instrucciones_unicas)
    embeddings_texto_unicos = encoder.encode_text(tokens_unicos).detach().cpu()
    mapa_texto = dict(zip(instrucciones_unicas, embeddings_texto_unicos))

    try:
        for imagenes_static_batch, imagenes_gripper_batch, _, acciones_batch in loader_cache:
            fin = inicio + len(imagenes_static_batch)
            particiones_batch = tabla_cache["particion"].iloc[inicio:fin].to_numpy()
            emb_static = encoder.encode_image(imagenes_static_batch).detach().cpu()
            emb_gripper = encoder.encode_image(imagenes_gripper_batch).detach().cpu()
            emb_textos = torch.stack([
                mapa_texto[instruccion]
                for instruccion in tabla_cache["instruccion"].iloc[inicio:fin]
            ])

            for nombre in np.unique(particiones_batch):
                mascara = torch.from_numpy(particiones_batch == nombre)
                acumulados[nombre]["imagenes_static"].append(emb_static[mascara])
                acumulados[nombre]["imagenes_gripper"].append(emb_gripper[mascara])
                acumulados[nombre]["textos"].append(emb_textos[mascara])
                acumulados[nombre]["acciones"].append(acciones_batch[mascara].cpu())
            inicio = fin
    finally:
        dataset_cache.cerrar_video()

    resultado = {}
    for nombre, valores in acumulados.items():
        emb_static = torch.cat(valores["imagenes_static"]).float()
        emb_gripper = torch.cat(valores["imagenes_gripper"]).float()
        emb_textos = torch.cat(valores["textos"]).float()
        if NORMALIZAR_EMBEDDINGS:
            emb_static = F.normalize(emb_static, dim=-1)
            emb_gripper = F.normalize(emb_gripper, dim=-1)
            emb_textos = F.normalize(emb_textos, dim=-1)
        resultado[nombre] = (
            emb_static.numpy(),
            emb_gripper.numpy(),
            emb_textos.numpy(),
            torch.cat(valores["acciones"]).numpy(),
        )
    return resultado

if CACHEAR_EMBEDDINGS:
    RUTA_CACHE.mkdir(parents=True, exist_ok=True)
    embeddings_por_particion = extraer_embeddings_secuenciales(tablas)
    for nombre, (imagenes_static, imagenes_gripper, textos, acciones) in embeddings_por_particion.items():
        salida = RUTA_CACHE / f'{nombre}.npz'
        np.savez_compressed(salida, imagenes_static=imagenes_static, imagenes_gripper=imagenes_gripper, textos=textos, acciones=acciones)
        print(f'{nombre}: {imagenes_static.shape[0]:,} muestras guardadas en {salida}')
else:
    print('CACHEAR_EMBEDDINGS=False: se omite el recorrido completo.')

train: 190,212 muestras guardadas en C:\TFM_Codigo\TFM-VLA\data\cache_embeddings\train.npz
validation: 23,760 muestras guardadas en C:\TFM_Codigo\TFM-VLA\data\cache_embeddings\validation.npz
test: 23,826 muestras guardadas en C:\TFM_Codigo\TFM-VLA\data\cache_embeddings\test.npz


## 7. Resumen

Cada muestra queda representada por un embedding `static`, uno `gripper`, uno textual en el mismo espacio multimodal y su acción normalizada.

In [7]:
print('Pipeline CLIP completado correctamente.')
print(f'Dimensión visual static: {embeddings_static.shape[-1]}')
print(f'Dimensión visual gripper: {embeddings_gripper.shape[-1]}')
print(f'Dimensión textual: {embeddings_texto.shape[-1]}')
print(f'Cache activada: {CACHEAR_EMBEDDINGS}')

Pipeline CLIP completado correctamente.
Dimensión visual static: 512
Dimensión visual gripper: 512
Dimensión textual: 512
Cache activada: True


## Resumen de resultados

Este notebook utiliza la configuración centralizada, crea las carpetas de salida necesarias y deja los artefactos del experimento en una carpeta propia.